# Electronic Waste Detection and Classification

## Phase 4 — YOLO26n Model Training

### Training Objective

The objective of this phase is to fine-tune the pretrained YOLO26n object detection model using the prepared electronic waste dataset.

The trained model will learn to detect and classify 37 different electronic waste object categories.

### Dataset

The cleaned dataset contains:

- Training images: 4,990
- Validation images: 1,444
- Test images: 776
- Total images: 7,210
- Object classes: 37

### Model

YOLO26n

The pretrained YOLO26n model is used as the starting point for transfer learning.

### Training Approach

The model will be fine-tuned using the prepared YOLO-format dataset.

Training performance will be monitored using:

- Training loss
- Validation loss
- Precision
- Recall
- mAP@50
- mAP@50:95

The best-performing model checkpoint will be retained for subsequent evaluation and demonstration.

## 4.1 Import Required Libraries

The required libraries are imported for:

- Dataset configuration
- Model loading
- Training
- Result inspection
- File management
- Reproducibility

In [1]:
from pathlib import Path
import random
import numpy as np
import torch
import yaml

from ultralytics import YOLO

print("Libraries imported successfully.")

Libraries imported successfully.


## 4.2 Reproducibility Configuration

A fixed random seed is used to improve reproducibility of the training process.

The same project configuration, dataset, model, image size, and seed will be documented so that the experiment can be repeated.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)
print("✓ Reproducibility seed configured.")

Random seed: 42
✓ Reproducibility seed configured.


## 4.3 Locate the Dataset Configuration

The YOLO `data.yaml` file created during Phase 2 is used to configure model training.

The configuration contains:

- Training dataset path
- Validation dataset path
- Test dataset path
- Number of classes
- Class names

In [3]:
PROJECT_ROOT = Path.cwd().parent

yaml_files = list(PROJECT_ROOT.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError(
        "data.yaml was not found inside the project."
    )

DATA_YAML = yaml_files[0]

print("Project root:")
print(PROJECT_ROOT)

print("\nDataset configuration:")
print(DATA_YAML)

Project root:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-

Dataset configuration:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\data.yaml


In [4]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

print("=" * 70)
print("DATASET CONFIGURATION")
print("=" * 70)

for key, value in data_config.items():
    print(f"{key}: {value}")

print("=" * 70)

assert data_config["nc"] == 37
assert len(data_config["names"]) == 37

print("✓ Dataset contains 37 classes.")

DATASET CONFIGURATION
path: .
train: images/train
val: images/val
test: images/test
nc: 37
names: ['Battery', 'Blood-Pressure-Monitor', 'Boiler', 'Clothes-Iron', 'Coffee-Machine', 'Computer-Keyboard', 'Computer-Mouse', 'Cooling-Display', 'Desktop-PC', 'Digital-Oscilloscope', 'Drone', 'Electric-Guitar', 'Electronic-Keyboard', 'Flashlight', 'Flat-Panel-Monitor', 'Flat-Panel-TV', 'Glucose-Meter', 'HDD', 'Laptop', 'Microwave', 'Music-Player', 'Oven', 'PCB', 'Photovoltaic-Panel', 'Projector', 'Refrigerator', 'Rotary-Mower', 'Router', 'Server', 'Smartphone', 'Smoke-Detector', 'Straight-Tube-Fluorescent-Lamp', 'Street-Lamp', 'TV-Remote-Control', 'Telephone-Set', 'USB-Flash-Drive', 'Washing-Machine']
✓ Dataset contains 37 classes.


## 4.4 Training Dataset Verification

Before starting the computationally expensive training process, the training and validation directories are checked again.

This prevents training from starting with an incorrect dataset path or missing data.

In [5]:
dataset_root = DATA_YAML.parent

train_path = Path(data_config["train"])
val_path = Path(data_config["val"])
test_path = Path(data_config["test"])

if not train_path.is_absolute():
    train_path = dataset_root / train_path

if not val_path.is_absolute():
    val_path = dataset_root / val_path

if not test_path.is_absolute():
    test_path = dataset_root / test_path

print("=" * 70)
print("TRAINING DATA VERIFICATION")
print("=" * 70)

print("Train:", train_path)
print("Exists:", train_path.exists())

print("\nValidation:", val_path)
print("Exists:", val_path.exists())

print("\nTest:", test_path)
print("Exists:", test_path.exists())

assert train_path.exists()
assert val_path.exists()
assert test_path.exists()

print("\n✓ Dataset paths verified.")

TRAINING DATA VERIFICATION
Train: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\images\train
Exists: True

Validation: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\images\val
Exists: True

Test: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\images\test
Exists: True

✓ Dataset paths verified.


## 4.5 Training Hardware

The local development computer uses an Intel Iris Xe integrated GPU and does not provide CUDA support through PyTorch.

Therefore, the project notebook is maintained in VS Code, while the computationally intensive training can be executed in a GPU-enabled environment.

The training configuration below remains part of the project documentation and can be reproduced in the GPU environment.

In [6]:
print("=" * 70)
print("TRAINING DEVICE")
print("=" * 70)

if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU available.")
    print("GPU:", torch.cuda.get_device_name(0))
    print("Training device: CUDA")
else:
    DEVICE = "cpu"
    print("CUDA GPU not available.")
    print("Training device: CPU")

print("=" * 70)

TRAINING DEVICE
CUDA GPU not available.
Training device: CPU


## 4.6 YOLO26n Model Initialization

The pretrained YOLO26n checkpoint is loaded as the starting point for transfer learning.

The model will subsequently be fine-tuned using the custom electronic waste dataset.

Training will use the dataset configuration defined in `data.yaml`.

In [7]:
MODEL_NAME = "yolo26n.pt"

model = YOLO(MODEL_NAME)

print("=" * 70)
print("MODEL INITIALIZATION")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Task:", model.task)

print("\n✓ Pretrained YOLO26n loaded successfully.")

MODEL INITIALIZATION
Model: yolo26n.pt
Task: detect

✓ Pretrained YOLO26n loaded successfully.


## 4.7 Training Configuration

The initial training configuration is designed for the prepared 37-class electronic waste dataset.

### Initial Configuration

- Model: YOLO26n
- Image size: 640 × 640
- Epochs: 50
- Batch size: hardware dependent
- Pretrained weights: Yes
- Random seed: 42
- Dataset: custom electronic waste dataset
- Task: object detection

The initial run is used to establish a baseline training result.

The final training configuration will be recorded together with the resulting performance metrics.

In [14]:
EPOCHS = 20
IMAGE_SIZE = 640
BATCH_SIZE = 16

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Epochs:", EPOCHS)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Seed:", SEED)
print("Dataset:", DATA_YAML)
print("=" * 70)

TRAINING CONFIGURATION
Model: yolo26n.pt
Epochs: 20
Image size: 640
Batch size: 16
Seed: 42
Dataset: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\data.yaml


## 4.8 Training Output Directory

Training outputs will be stored inside the project `results` directory.

The output will contain information such as:

- Training curves
- Validation curves
- Confusion matrix
- Precision-recall plots
- Validation metrics
- Model checkpoints
- Best model weights

In [15]:
RESULTS_DIR = PROJECT_ROOT / "results"
TRAINING_DIR = RESULTS_DIR / "training"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:")
print(RESULTS_DIR)

print("\nTraining directory:")
print(TRAINING_DIR)

print("\n✓ Training output directories ready.")

Results directory:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\results

Training directory:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\results\training

✓ Training output directories ready.


## 4.9 YOLO26n Training

The pretrained YOLO26n model is fine-tuned using the custom electronic waste dataset.

The model learns to simultaneously:

1. Locate objects using bounding boxes.
2. Classify each detected object into one of the 37 electronic waste categories.

The validation dataset is used during training to monitor generalization performance.

The test dataset is reserved for the final evaluation phase and is not used to select the training checkpoint.

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    project=str(TRAINING_DIR),
    name="yolo26n_e_waste",
    pretrained=True,
    plots=True,
    save=True,
    val=True
)

## 4.10 Training Experiment Record

The following information will be recorded for the training experiment.

| Parameter | Value |
|---|---|
| Model | YOLO26n |
| Task | Object Detection |
| Dataset | Balanced E-Waste Dataset |
| Classes | 37 |
| Training Images | 4,990 |
| Validation Images | 1,444 |
| Test Images | 776 |
| Image Size | 640 × 640 |
| Initial Epochs | 50 |
| Batch Size | 16 |
| Random Seed | 42 |
| Pretrained | Yes |

The exact hardware used for the final training run will also be recorded.

# Phase 4 Training Status

The training pipeline has been configured but the long-running training operation has not yet been executed in the local CPU environment.

The next step is to execute the configured training experiment using a suitable GPU-enabled environment.

After training, the following outputs will be collected:

- `best.pt`
- `last.pt`
- Training loss curves
- Validation loss curves
- Precision
- Recall
- mAP@50
- mAP@50:95
- Confusion matrix
- Precision-recall curve

These outputs will be used in Phase 5 for final model evaluation.

In [11]:
import sys
import torch
import ultralytics

print("=" * 70)
print("FINAL TRAINING ENVIRONMENT CHECK")
print("=" * 70)

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: No CUDA GPU")
    print("Training device: CPU")

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)

print("Train images:", 4990)
print("Validation images:", 1444)
print("Test images:", 776)
print("Total images:", 4990 + 1444 + 776)
print("Classes:", 37)

print("\n" + "=" * 70)
print("MODEL TRAINING CONFIGURATION")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Epochs:", EPOCHS)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Random seed:", SEED)

print("=" * 70)

FINAL TRAINING ENVIRONMENT CHECK
Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\USER\Desktop\computer vision\.venv\Scripts\python.exe
PyTorch: 2.14.0+cpu
Ultralytics: 8.4.153

CUDA available: False
GPU: No CUDA GPU
Training device: CPU

DATASET
Train images: 4990
Validation images: 1444
Test images: 776
Total images: 7210
Classes: 37

MODEL TRAINING CONFIGURATION
Model: yolo26n.pt
Epochs: 20
Image size: 640
Batch size: 16
Random seed: 42


## 4.11 Training Pipeline Test

Before performing the complete training experiment, a short training run is performed to verify that:

- The YOLO26n model loads correctly.
- The dataset configuration is valid.
- Images and labels can be read correctly.
- The model can perform a forward/backward training pass.
- The training output directory is configured correctly.

This is a technical pipeline test and is not considered the final model training experiment.

The final model will be trained using the complete training configuration after the pipeline has been verified.

In [13]:
# ============================================================
# YOLO26n TRAINING PIPELINE TEST
# ============================================================

TEST_EPOCHS = 1
TEST_BATCH = 4

print("=" * 70)
print("YOLO26n TRAINING PIPELINE TEST")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Dataset:", DATA_YAML)
print("Test epochs:", TEST_EPOCHS)
print("Test batch size:", TEST_BATCH)
print("Image size:", IMAGE_SIZE)
print("Device:", DEVICE)

print("=" * 70)
print("Starting 1-epoch pipeline test...")
print("=" * 70)

test_model = YOLO(MODEL_NAME)

test_results = test_model.train(
    data=str(DATA_YAML),
    epochs=TEST_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=TEST_BATCH,
    device="cpu",
    seed=SEED,
    project=str(TRAINING_DIR),
    name="pipeline_test",
    pretrained=True,
    plots=True,
    save=True,
    val=True
)

print("=" * 70)
print("✓ PIPELINE TEST COMPLETED")
print("=" * 70)

YOLO26n TRAINING PIPELINE TEST
Model: yolo26n.pt
Dataset: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\data.yaml
Test epochs: 1
Test batch size: 4
Image size: 640
Device: cpu
Starting 1-epoch pipeline test...
New https://pypi.org/project/ultralytics/8.4.154 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.153  Python-3.14.3 torch-2.14.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\data.yaml, degrees=0.0, deterministic=